# LULC Segmentation from Sentinel-1: final iteration

The final training notebook of the project. It trains **one model**, `final_model.pt`, for the prototype: 12 monthly land-cover maps of rural land anywhere on Earth, from Sentinel-1 radar only.

It is `Notebooks/2nd_iteration.ipynb` (the notebook of `Iter2_Test_1_Full`, the model used until now) with the additions of the 4th iteration:
- **more training data where the prototype failed:** radar of 2022-2024 for the training sites, and 17 new sites with bare ground outside deserts, small tropical fields, bare rock and pure bare / crop tiles (`data_pipeline/config.py`, `ITER4_SITES`);
- **a class bias** for Water and Man-made, tuned on val, stored in the model card and applied by the prototype.

Everything else is unchanged from `Iter2_Test_1_Full`, because the experiments showed it is the best choice (next section).

## Why we believe this configuration works best

Every choice below was tested in an earlier iteration; the evidence is in `reports/`. The rule throughout: choose by **val**, report **test**, and count a difference only if it is larger than the run-to-run noise (val ±0.006, rural test ±0.03 between identical runs, `reports/2026-11-05 3rd iteration experiment results.md` section 3).

| Choice | Evidence | Source |
|---|---|---|
| **Input `month_year`**: target month + 9 order-independent seasonal statistics (12 channels) | Seasonal statistics beat PCA on test (0.742 vs 0.617 mIoU): PCA loadings are tied to calendar months and fail across hemispheres. With the statistics, one month cannot flip the class (Black Forest winter failure fixed). | Test run 6; 2026-10-31 Black Forest report |
| **Radar: terrain-flattened gamma0, no median filter** | VV correlated 0.76 with the slope before flattening, -0.09 after; the median blurred edges. | 2026-10-31 report, section 6.2 |
| **7 classes, rural sites** | Wetland labels are only 30-45 % correct; city pixels inflated the 1st-iteration score (54 % of test pixels). | 2026-10-31 report, sections 6.1-6.4 |
| **U-Net with an ImageNet ResNet-34 encoder** | Five encoders (5.9-32.6 M parameters) were within noise of each other; ResNet-34 is the fastest of the top group. ImageNet pretraining is the largest model effect measured: without it, -0.06 val and -0.09 rural mIoU. | 3rd iteration, section 4.1 |
| **AdamW + cross-entropy + Lovász** | SGD: no gain beyond noise. Without Lovász: -0.016 val and -0.012 rural mIoU, Snow & ice fell most. | 3rd iteration, section 4.2 |
| **WorldCover on all pixels, disagreement weight 0.5** | Agreement-only training scores higher on agreement pixels but loses 0.023 on all pixels (it ignores edges and makes blob-shaped maps). Full weight is worse on val. Soft bootstrapping is promising but within noise. | 3rd iteration, section 4.3 |
| **Month dropout 0.3, 3-month smoothing** | Flicker between months 2.4 % → 0.9 % with smoothing; the stable Black Forest and Thane maps through winter and monsoon. | 2026-10-31 report, section 11.2; 2026-11-02 report |
| **40 epochs, early stopping (patience 10), EMA** | The 40-epoch run beat both 15-epoch runs by 0.04-0.055 rural mIoU, more than the noise; val peaks around epoch 22-30. | 2026-10-31 report, section 11.2 |
| **New: later years (2022-2024)** | The prototype runs on 2022-2025 radar, which has about half the scenes per month (Sentinel-1B lost in December 2021). Training on those years shows the model sparser composites and other weather years. Pixels whose land cover changed after 2021 get the lower weight automatically, because Dynamic World of that year disagrees with WorldCover 2021. | 2026-11-05 data pipeline |
| **New: sites for the observed failures** | Every bare-ground site was a desert, so eroded laterite (Madagascar highlands) was mapped as Grassland / Crop fields; 18 of 22 crop sites are large flat fields; bright bare rock looked like Man-made. The model can only learn what is in the data: the architecture experiments showed that the errors come from data and labels, not from the network. | Prototype runs 2026-11-05; 3rd iteration |
| **Class bias: tested, off** | Water is over-predicted (test precision 0.56-0.59, recall 0.74), but in the 4th iteration no offset for Water or Man-made improved val mIoU: the false Water pixels are places where radar cannot tell water from smooth dark ground, not a threshold problem. The search stays in section 9.6 with `BIAS_CLASSES = []`. | 4th iteration, section 9.4 |

## Why we added more data

**What went wrong with `Iter2_Test_1_Full`.** In the prototype (Madagascar highlands, a rural test site the model never trained on) eroded red laterite and bare hills were mapped as **Grassland and Crop fields**, crop fields were confused with grassland and bare ground, some bright bare rock became **Man-made**, and **Water** appeared on dark, smooth land. On the test set the same weak spots show: Crop fields IoU 0.656 and Grassland 0.682 (the most confused pair), Water 0.492, Africa the second-weakest region (0.492). The 3rd iteration ruled out the network as the cause: five encoders made the same mistakes. The cause is in the data:

| Gap in the training data | Consequence |
|---|---|
| All 7 bare-ground sites were deserts (Sahara, Namib, Taklamakan ...) | the model learned "bare = sand or rock desert"; bare soil in humid, hilly land looked like grass or fields |
| 18 of 22 crop sites are large flat fields (Iowa, Punjab, Ukraine ...) | small, irregular tropical fields between grass and trees were not recognised as crops |
| no bare rock outside cities | bright, rough rock was taken for buildings |
| only 2021 radar (Sentinel-1A + 1B) | the prototype runs on 2022-2025 radar with half the scenes per month, which the model never saw |

**What is added, and why each kind.**
- **Mixed tiles where the errors happen** (13 sites, `need` = a minimum share of the target class): eroded tropical highlands, mines, a degraded highland, smallholder farms, terraces and paddies, bare rock. They teach the boundaries: bare hill next to grass, a small field next to forest.
- **Pure-class tiles** (4 sites, one class covers at least 60 % of the tile): two bare salt pans (Etosha, Uyuni) and two irrigated crop schemes (Gezira, central Thailand). They show the model what a class alone looks like to the radar, so it can tell the classes apart inside a mixed tile. The smooth salt pans also look dark like water to the radar, so they are direct counter-examples for false Water.
- **Later years** (2022-2024) of every training site except towns and glaciers: the same places under other weather and with the sparser post-2021 radar.

**Class balance.** The classes are the same 7 as before, with the same mapping from WorldCover and Dynamic World (`data_pipeline/config.py`), so the model card, the prototype and all scores stay compatible.
- Later years do not shift the balance: a window that exists in several years shares its sampling weight between its years. Without this, towns and glaciers (no later years) would lose three quarters of their weight, and Man-made and Snow & ice would be under-trained.
- The new sites raise the share of Bare ground and Crop fields among the drawn pixels on purpose, because those are the classes that failed; section 5 prints the expected share per class for the reference and for this run.
- Class weights (loss) and the class-aware sampler are those of `Iter2_Test_1_Full`, computed from the reference data, so a rare class is not suddenly over- or under-weighted.

**Validation and test.** Splits are by site and unchanged: 21 val and 17 test sites, all 2021, never trained on. Every new site is a training site at least 30 km from every val/test site (`config.leakage()`; the only closer pairs are two sites of the 2nd iteration, 28.8 km and 19.7 km, unchanged). The model is chosen by val, test is only reported, so the scores stay comparable with all earlier runs. The test set contains the Madagascar highlands and Kano farmland, so a fix of the bare / crop confusion shows in the Africa score and in the Crop fields and Bare ground IoU. What val/test cannot show: maps of 2022+ radar (they are 2021 only); check those in the prototype with a 2024 or 2025 start.

**Will it predict better?** Expected, not guaranteed: more examples of the failing cases usually reduce exactly those errors, and the class bias targets the Water false alarms. The 3rd iteration showed that rural test scores vary by about ±0.03 between identical runs, so judge the result by val (and two seeds in `Notebooks/4th_iteration_multiyear.ipynb`), by the Africa, Crop fields and Bare ground scores, and by the prototype in Madagascar.

**What is not yet proven.** The two data additions are measured in `Notebooks/4th_iteration_multiyear.ipynb` (reference vs new sites vs later years vs both, 2 seeds, unchanged 2021 val/test). Set `TRAIN_YEARS` and `USE_NEW_SITES` below to the winner of that notebook. If neither addition beats the reference by more than the noise, set `TRAIN_YEARS = (2021,)` and `USE_NEW_SITES = False`: this notebook then trains exactly the `Iter2_Test_1_Full` configuration, plus the class bias.

**Expected level** (from the 3rd iteration): rural test mIoU about 0.66 ± 0.03 for this configuration on the 2021 test sites; the data additions target regions and conditions the test set barely covers (Madagascar-type highlands, 2022+ radar), so check them in the prototype as well.

**Roadmap**

1. Settings and hardware check
2. Load the data (2021 + later years + new sites)
3. Explore the data
4. Input features
5. Datasets, augmentation and class balancing
6. The CNN model
7. Loss and metrics
8. Training
9. Results (curves, 12-month test, confusion, regions, monthly maps, class bias)
10. Export for the prototype
11. Save the run


## Glossary: new terms in this iteration

All other terms are explained in the glossary of `Notebooks/1st_iteration.ipynb`.

| Term | Meaning |
|---|---|
| sigma0, gamma0 | two ways to normalise radar backscatter. sigma0 is per unit of flat ground area; gamma0 is per unit of area seen by the radar beam, which removes most of the dependence on the incidence angle |
| terrain flattening | correcting gamma0 for the local slope: a slope facing the satellite receives more energy per pixel and looks brighter, a slope facing away looks darker. Without it, hillsides look like buildings or water |
| layover, radar shadow | on very steep slopes towards the satellite the echoes of top and bottom overlap (layover); behind steep slopes the radar sees nothing (shadow). Both are masked |
| mosaic site | a 10 km tile that mixes several land-cover classes (fields, meadows, forest, hamlets), checked with WorldCover before download |
| agreement pixel | a pixel where WorldCover and Dynamic World give the same class. Training uses every WorldCover pixel but trusts agreement pixels more; the main scores use agreement pixels only |
| seasonal statistics | 9 numbers per pixel that summarise its 12 months without depending on the month order (mean, 2nd lowest and 2nd highest month, spread, water months) |
| month dropout | during training the monthly input is blanked at random, so the model learns to fall back on the yearly statistics instead of trusting one odd month |
| temporal smoothing | averaging the class probabilities of months t-1, t and t+1 before choosing the class. Short one-month errors disappear; changes that last 2 months or more stay |
| flicker | the share of pixels whose predicted class changes from one month to the next. The labels are the same all year, so for most land every change is a false change. Lower is better |
| rural test mIoU | mIoU on the test sites that are not towns. **The main score of this iteration** |
| later years | radar of 2022-2024 for the training sites, paired with the WorldCover 2021 label (only 2020 and 2021 exist); Dynamic World of the same year decides which pixels count as agreement |
| class bias (`class_logit_bias`) | an offset per class added to the log-probabilities before the class is chosen; a negative Water offset means "choose Water only when it is clearly more likely" |


## 1. Setup, settings and hardware check

All settings are in the next cell. The model and training settings are those of `Iter2_Test_1_Full` (justified in the introduction); only the data and the class bias are new.

- `TRAIN_YEARS`: radar years of the training windows (val and test are always 2021). Default: 2021-2024.
- `USE_NEW_SITES`: train on the 17 sites added in the 4th iteration. Default: yes.
- Both should match the winner of `Notebooks/4th_iteration_multiyear.ipynb`; `(2021,)` + `False` reproduces `Iter2_Test_1_Full`.
- Normalisation and class weights come from the 2021 windows of the earlier sites, as in `Iter2_Test_1_Full`, so the comparison with it is fair.
- `BIAS_CLASSES`, `BIAS_GRID`: class bias search on val (section 9.6).
- `RUN_NAME` is the results folder `Weights/<RUN_NAME>/`; an existing folder is never overwritten.


In [ ]:
RUN_NAME = "Final_Iteration"
NOTEBOOK = "Notebooks/final_iteration.ipynb"

TRAIN_YEARS = (2021, 2022, 2023, 2024)   # radar years of the training windows; (2021,) = Iter2_Test_1_Full data
USE_NEW_SITES = True                     # the 17 sites of the 4th iteration (bare ground, smallholder fields, rock, pure-class tiles)

ALL_VARIANTS = {"month_year": dict(n_months=1)}
FINAL_VARIANT = "month_year"   # target month + 9 seasonal statistics: the input of Iter2_Test_1_Full
RUN = [FINAL_VARIANT]
DEV = False
MAX_EPOCHS = 40

VAL_MONTHS = list(range(12))
VAL_EVERY = 2
PATIENCE = 10
SAMPLES_PER_EPOCH = 2500
LEARNING_RATE = 6e-4
WARMUP_EPOCHS = 1
EMA_DECAY = 0.99
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
DROPOUT = 0.2
MONTH_DROP = 0.3        # lean on the yearly statistics, not on one month (also covers empty months after 2021)
DISAGREE_WEIGHT = 0.5   # loss weight where WorldCover and Dynamic World disagree (agreement pixels: 1)
LOVASZ_WEIGHT = 0.5
LOVASZ_SAMPLE = 0.25
CROP_JITTER = 128
SEED = 42

BIAS_CLASSES = []                                                # 4th iteration: no offset improved val; add ["Water", "Man-made"] to search again
BIAS_GRID = [-1.5, -1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5]     # log-probability offsets tried per class
BIAS_MIN_GAIN = 0.002                                            # keep an offset only if val mIoU rises by more

VARIANTS = {name: ALL_VARIANTS[name] for name in RUN}
print(f"final run | variant {FINAL_VARIANT} | training years {TRAIN_YEARS} | new sites {USE_NEW_SITES} | "
      f"max epochs {MAX_EPOCHS}")


In [ ]:
import subprocess, sys
try:
    import rasterio
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
    import rasterio

import json, math, os, shutil, time, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# The notebook lives in Notebooks/; data and weights live at the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
DATA_DIRS = [ROOT / "Dataset_2nd_iteration", Path("../input")]
OUT = ROOT / "Weights" / RUN_NAME
if OUT.exists() and any(OUT.iterdir()):
    OUT = OUT.with_name(f"{RUN_NAME}_{time.strftime('%Y%m%d_%H%M%S')}")
    print(f"Weights/{RUN_NAME} already has files, saving to {OUT} instead")


def s1_preprocessing(root):
    first = next((root / "s1").glob("*.tif"), None)
    if first is None:
        return ""
    with rasterio.open(first) as src:
        return src.tags().get("preprocessing", "")


def find_data_root():
    """The first dataset with a patch index whose radar tiles are terrain-flattened (2nd-iteration pipeline)."""
    for base in DATA_DIRS:
        if base.exists():
            for hit in sorted(base.rglob("train_patches.csv")):
                root = hit.parent.parent
                if "gamma0" in s1_preprocessing(root):
                    return root
                print("skipping (not built by the 2nd-iteration pipeline):", root)
    raise FileNotFoundError("Dataset_2nd_iteration not found: build it with data_pipeline/ (see data_pipeline/README.md)")


DATA = find_data_root()
S1_PREPROCESSING = s1_preprocessing(DATA)
VARIANT_DIR = OUT / "variants"  # one checkpoint per trained variant; the delivered model is OUT / final_model.pt
VARIANT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA, "| output:", OUT.resolve())
print("radar preprocessing:", S1_PREPROCESSING)

### Hardware check

Unchanged from the 1st iteration (section 1 there explains every rule): bfloat16 on Ampere and newer, float16 + `GradScaler` on T4; batch 64 with ≥ 24 GB GPU memory, else 32; no DataLoader workers when `/dev/shm` is small.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
SHM_GB = shutil.disk_usage("/dev/shm").total / 1e9 if Path("/dev/shm").exists() else None

if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    GPU_NAME, GPU_MEM_GB, CAPABILITY = props.name, props.total_memory / 1e9, (props.major, props.minor)
else:
    GPU_NAME, GPU_MEM_GB, CAPABILITY = "none", 0.0, (0, 0)

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if CAPABILITY >= (8, 0) else torch.float16
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

BATCH = 64 if GPU_MEM_GB >= 24 else 32
GPU_TRAIN_RESERVE_GB = 12 if BATCH == 64 else 7

# DataLoader workers hand batches over through shared memory (/dev/shm). Docker containers often
# give it only 64 MB, which kills the workers with "Bus error"; then load in the main process.
NUM_WORKERS = 0 if SHM_GB is not None and SHM_GB < 1 else min(8, CPU_COUNT)

print(f"GPU: {GPU_NAME} | {GPU_MEM_GB:.0f} GB | compute capability {CAPABILITY[0]}.{CAPABILITY[1]}")
print(f"CPU cores: {CPU_COUNT} | /dev/shm: {'n/a' if SHM_GB is None else f'{SHM_GB:.2f} GB'}")
print(f"mixed precision: {str(AMP_DTYPE).split('.')[-1] if USE_AMP else 'off'}{' + GradScaler' if USE_SCALER else ''}"
      f" | batch {BATCH} | DataLoader workers {NUM_WORKERS} (only used if the data does not fit on the GPU)")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| rasterio", rasterio.__version__)

### Dataset constants

- **Radar values** are uint8 codes: `dB = code * 0.2 - 35`, code 0 = no data (also layover and shadow).
- **Classes** (7; 1st iteration: 8):

| Class | Includes | WorldCover |
|---|---|---|
| Man-made | buildings, roads, paved yards, railways | 50 |
| Crop fields | land ploughed and harvested each year | 40 |
| Trees | forest, orchards, plantations, mangrove | 10, 95 |
| Grassland | pasture, meadows, steppe, shrub and scrub, marsh | 20, 30, 90 |
| Bare ground | desert, rock, sand, sparse vegetation | 60, 100 |
| Water | lakes, rivers, sea | 80 |
| Snow & ice | glaciers, permanent snow | 70 |

- **Label code** (one int16 array per tile, so labels and agreement travel together through cropping, flips and the GPU):
  - `0 ... 6`: class index, WorldCover and Dynamic World agree;
  - `7 ... 13`: class index + 7, only WorldCover (the maps disagree);
  - `-1`: no WorldCover data (ignored).

  `split_label` turns a code into the class index and the agreement mask.

In [ ]:
CLASS_NAMES = ["Man-made", "Crop fields", "Trees", "Grassland", "Bare ground", "Water", "Snow & ice"]
CLASS_COLORS = ["#e31a1c", "#ffd92f", "#1a9850", "#a6d96a", "#bf812d", "#2c7fb8", "#dfe9f0"]
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
N_CLASSES = len(CLASS_NAMES)
IGNORE = -1
PATCH = 256
DB_MIN, DB_STEP = -35.0, 0.2

DB_LUT = (np.arange(256, dtype=np.float32) * DB_STEP + DB_MIN)
DB_LUT[0] = np.nan

CMAP = ListedColormap(["#000000"] + CLASS_COLORS)
LEGEND = [Patch(color=c, label=n) for n, c in zip(["no label"] + CLASS_NAMES, ["#000000"] + CLASS_COLORS)]


def label_classes(y):
    """numpy label codes -> class index (IGNORE where no label)."""
    return np.where(y >= 0, y % N_CLASSES, IGNORE)


def split_label(y):
    """torch label codes -> (class index or IGNORE, mask of pixels where both label maps agree)."""
    target = torch.where(y >= 0, y % N_CLASSES, torch.full_like(y, IGNORE))
    return target, (y >= 0) & (y < N_CLASSES)

## 2. Load the data

Same layout as the 2nd iteration: 1024 × 1024 px tiles (10.24 km), 12 months × 2 polarisations, splits by site. Train keeps the full tiles in memory (random crops around the indexed windows); val and test use the fixed windows of 2021. The label is WorldCover (band 2 of the label file) on every pixel, with the consensus band (band 1) marking agreement.

Added for the final iteration:
- `patches/train_extra_patches.csv`: windows of the training sites in later years (`s1_<year>/`, `labels_<year>/`); a training tile is keyed by `(site, year)`. Only `TRAIN_YEARS` are loaded.
- `iteration` column: 4 = sites added in the 4th iteration. With `USE_NEW_SITES = False` they are loaded but never drawn.
- `REF` marks the 2021 windows of the earlier sites: normalisation and class weights come from them.


In [ ]:
SPLITS = ["train", "val", "test"]
index = {s: pd.read_csv(DATA / "patches" / f"{s}_patches.csv") for s in SPLITS}
for s in SPLITS:
    if "iteration" not in index[s]:
        index[s]["iteration"] = 2   # index written before the 4th iteration
    rural = index[s][index[s].focus != "urban"].site.nunique()
    print(f"{s:5}: {len(index[s]):5} patches from {index[s].site.nunique():3} sites ({rural} rural)")
extra_path = DATA / "patches" / "train_extra_patches.csv"
extra = pd.read_csv(extra_path) if extra_path.exists() else pd.DataFrame(columns=["site", "year", "iteration", "row", "col"])
extra = extra[extra.year.isin([y for y in TRAIN_YEARS if y != 2021])]
missing = sorted(set(TRAIN_YEARS) - {2021} - set(extra.year.astype(int)))
if missing:
    print(f"warning: no training windows for {missing}: build them with data_pipeline/build_dataset.py")
# 2021 windows first, in index order (positions = index["train"] rows), then the later years
train_all = pd.concat([index["train"].assign(year=2021), extra], ignore_index=True)
print("training windows per year and site group (iteration 2 = earlier sites, 4 = new sites):")
display(pd.crosstab(train_all.year, train_all.iteration))


def read_site(site, year=2021):
    s1_dir, label_dir = ("s1", "labels") if year == 2021 else (f"s1_{year}", f"labels_{year}")
    with rasterio.open(DATA / s1_dir / f"{site}.tif") as src:
        s1 = src.read().reshape(12, 2, src.height, src.width)
    with rasterio.open(DATA / label_dir / f"{site}.tif") as src:
        consensus, worldcover = src.read(1), src.read(2)
    label = worldcover.astype(np.int16) - 1
    label[(worldcover > 0) & (consensus == 0)] += N_CLASSES
    return s1, label


TILE_X, TILE_Y = {}, {}
X, Y = {}, {}
t0 = time.time()
for s in ("val", "test"):
    df = index[s]
    X[s] = np.empty((len(df), 12, 2, PATCH, PATCH), dtype=np.uint8)
    Y[s] = np.empty((len(df), PATCH, PATCH), dtype=np.int16)
    for site, rows in df.groupby("site"):
        s1, label = read_site(site)
        for i, r, c in zip(rows.index, rows.row, rows.col):
            X[s][i] = s1[:, :, r:r + PATCH, c:c + PATCH]
            Y[s][i] = label[r:r + PATCH, c:c + PATCH]
for (site, year), _ in train_all.groupby(["site", "year"]):
    TILE_X[(site, int(year))], TILE_Y[(site, int(year))] = read_site(site, int(year))

TRAIN_ORIGINS = list(zip(zip(train_all.site, train_all.year.astype(int)), train_all.row, train_all.col))
ORIGIN_YEAR = train_all.year.astype(int).to_numpy()
ORIGIN_ITER = train_all.iteration.astype(int).to_numpy()
REF = (ORIGIN_YEAR == 2021) & (ORIGIN_ITER == 2)
REF_IDX = np.flatnonzero(REF)
USED = np.isin(ORIGIN_YEAR, TRAIN_YEARS) & ((ORIGIN_ITER == 2) | USE_NEW_SITES)


def train_window(i, dr=0, dc=0):
    site, r, c = TRAIN_ORIGINS[i]
    size = TILE_Y[site].shape[0]
    return site, int(np.clip(r + dr, 0, size - PATCH)), int(np.clip(c + dc, 0, size - PATCH))


def train_patch(i, dr=0, dc=0):
    site, r, c = train_window(i, dr, dc)
    return TILE_X[site][:, :, r:r + PATCH, c:c + PATCH], TILE_Y[site][r:r + PATCH, c:c + PATCH]


Y["train"] = np.stack([train_patch(i)[1] for i in range(len(TRAIN_ORIGINS))])
gb = (sum(t.nbytes for t in TILE_X.values()) + sum(x.nbytes for x in X.values())) / 1e9
print(f"loaded {len(TILE_X)} training tiles (site-years) in {time.time() - t0:.0f}s, {gb:.2f} GB of radar data; "
      f"{USED.sum()} of {len(TRAIN_ORIGINS)} training windows are used, {len(REF_IDX)} are reference windows")


## 3. Explore the data

**Class distribution and agreement.** The share of each class among WorldCover-labelled pixels, and how often Dynamic World agrees. Low agreement marks classes whose labels are uncertain (typically grassland and the edges between classes), which is why disagreement pixels get a lower loss weight.

In [ ]:
counts = {s: np.bincount(label_classes(Y[s])[Y[s] >= 0].ravel(), minlength=N_CLASSES) for s in SPLITS}
# class weights and normalisation come from the reference windows (2021, earlier sites), as in Iter2_Test_1_Full
Y_REF = Y["train"][REF_IDX]
counts["train_ref"] = np.bincount(label_classes(Y_REF)[Y_REF >= 0].ravel(), minlength=N_CLASSES)
Y_USED = Y["train"][USED]
counts["train_used"] = np.bincount(label_classes(Y_USED)[Y_USED >= 0].ravel(), minlength=N_CLASSES)
agree_counts = {s: np.bincount(Y[s][(Y[s] >= 0) & (Y[s] < N_CLASSES)].ravel(), minlength=N_CLASSES) for s in SPLITS}
dist = pd.DataFrame({f"{s} %": counts[s] / counts[s].sum() * 100 for s in SPLITS}, index=CLASS_NAMES)
dist["train used %"] = counts["train_used"] / counts["train_used"].sum() * 100
dist["train reference %"] = counts["train_ref"] / counts["train_ref"].sum() * 100
dist["train agreement %"] = agree_counts["train"] / np.maximum(counts["train"], 1) * 100
display(dist.round(1))
for s in SPLITS:
    print(f"{s}: {agree_counts[s].sum() / counts[s].sum() * 100:.1f}% of labelled pixels are agreement pixels")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
dist[[f"{s} %" for s in SPLITS]].plot.bar(ax=axes[0], color=["#4c72b0", "#dd8452", "#55a868"])
axes[0].set_ylabel("% of labelled pixels")
axes[0].set_title("Class distribution per split (WorldCover)")
axes[0].tick_params(axis="x", rotation=30)
all_patches = pd.concat([index[s].assign(split=s) for s in SPLITS], ignore_index=True)
pd.crosstab(all_patches.focus, all_patches.split)[SPLITS].plot.bar(ax=axes[1])
axes[1].set_ylabel("patches")
axes[1].set_title("Patches per site type and split")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**What the radar sees** (R = VV, G = VH, B = VV - VH; explained in the 1st iteration, section 3). With terrain flattening, hillsides in mosaic tiles should no longer show a bright / dark stripe pattern that follows the relief. Black in the agreement row marks pixels where the two label maps disagree: they cluster along boundaries.

In [ ]:
def false_colour(codes_2hw):
    vv, vh = DB_LUT[codes_2hw[0]], DB_LUT[codes_2hw[1]]
    rgb = np.stack([vv, vh, vv - vh], -1)
    for b, (lo, hi) in enumerate([(-20, 0), (-27, -7), (2, 15)]):
        rgb[..., b] = np.clip((rgb[..., b] - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(rgb)


focus_types = [f for f in ["mosaic", "cropland", "trees", "urban"] if f in set(index["train"].focus)]
examples = [index["train"][index["train"].focus == f].index[0] for f in focus_types]
fig, axes = plt.subplots(4, len(examples), figsize=(4 * len(examples), 16), squeeze=False)
for j, i in enumerate(examples):
    x, y = train_patch(i)
    axes[0, j].imshow(false_colour(x[0]))
    axes[0, j].set_title(f"{index['train'].site[i]} - January")
    axes[1, j].imshow(false_colour(x[6]))
    axes[1, j].set_title("July")
    axes[2, j].imshow(label_classes(y) + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
    axes[2, j].set_title("label (WorldCover)")
    axes[3, j].imshow((y >= 0) & (y < N_CLASSES), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    axes[3, j].set_title("agreement (white) / disagreement (black)")
for ax in axes.ravel():
    ax.axis("off")
fig.legend(handles=LEGEND, loc="lower center", ncol=9)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

## 4. Input features: monthly channels and seasonal statistics

**Monthly channels** (unchanged): VV, VH and VV - VH in dB per input month, standardised with train-only statistics; missing pixels become 0 (the channel mean). The month window wraps around the year (January uses November and December).

**Seasonal statistics** (every variant). Rural land-cover classes are defined by their yearly cycle: fields are ploughed, grow and are harvested; grassland stays green; forest is stable; water is dark all year. The statistics describe that cycle **without depending on the month order**, so they work the same in both hemispheres (the reason PCA was dropped, see the 1st iteration, section 5, and `reports/literature survey_reiterate.md`):

| Channel | Meaning | Helps with |
|---|---|---|
| VV mean, VH mean, VV-VH mean | the average signal over the year | city vs forest vs field vs water |
| VV low, VH low | 2nd darkest month (robust minimum) | water (always dark), snow and frozen ground (dark in winter only) |
| VV high, VH high | 2nd brightest month (robust maximum) | crops at full growth, buildings |
| VH std | how much the vegetation signal varies over the year | cropland (strong cycle) vs trees and grassland (weak) |
| water months | share of months with VV below -18 dB | permanent water (~1) vs seasonally flooded wetland (between 0 and 1) |

A winter month that looks like water can no longer flip a forest pixel on its own: the yearly statistics still say "forest". Month dropout (section 5) teaches the model to trust them.

All statistics are standardised with train statistics and computed **once** per tile (float16).

In [ ]:
def standardise(f, mean, std):
    return np.nan_to_num((f - mean[None, :, None, None]) / std[None, :, None, None], nan=0.0)


def window_months(month, n_months):
    return [(month - j) % 12 for j in range(n_months - 1, -1, -1)]


def take_months(codes, months):
    """codes (12, 2, H, W) -> (len(months), 2, H, W); an empty month list gives an empty array (year_stats)."""
    return codes[months] if months else codes[:0]


# normalisation from the reference windows, as in Iter2_Test_1_Full (the prototype reads it from the card)
sample = DB_LUT[np.stack([train_patch(i)[0][:, :, ::4, ::4] for i in REF_IDX[::4]])]
feats = np.stack([sample[:, :, 0], sample[:, :, 1], sample[:, :, 0] - sample[:, :, 1]], 0).reshape(3, -1)
CH_MEAN = np.nanmean(feats, axis=1).astype(np.float32)
CH_STD = np.nanstd(feats, axis=1).astype(np.float32)
del sample, feats
print("channel mean (VV, VH, VV-VH):", CH_MEAN.round(3))
print("channel std  (VV, VH, VV-VH):", CH_STD.round(3))

WATER_DB = -18.0
STAT_NAMES = ["VV mean", "VH mean", "VV-VH mean", "VV low", "VV high", "VH low", "VH high", "VH std", "water months"]
N_STATS = len(STAT_NAMES)


def stats_raw(codes):
    db = DB_LUT[codes]
    vv, vh = db[:, 0], db[:, 1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        vv_sorted = np.sort(np.where(np.isnan(vv), np.nanmean(vv, 0), vv), 0)
        vh_sorted = np.sort(np.where(np.isnan(vh), np.nanmean(vh, 0), vh), 0)
        n_valid = np.maximum((~np.isnan(vv)).sum(0), 1)
        out = np.stack([np.nanmean(vv, 0), np.nanmean(vh, 0), np.nanmean(vv - vh, 0),
                        vv_sorted[1], vv_sorted[-2], vh_sorted[1], vh_sorted[-2],
                        np.nanstd(vh, 0), (vv < WATER_DB).sum(0) / n_valid])
    return out.astype(np.float32)


def fit_mean_std(maps):
    flat = np.concatenate([m.reshape(m.shape[0], -1) for m in maps], 1)
    return np.nanmean(flat, 1).astype(np.float32), np.nanstd(flat, 1).astype(np.float32)


STAT_MEAN, STAT_STD = fit_mean_std([stats_raw(train_patch(i)[0][:, :, ::8, ::8])
                                    for i in REF_IDX[::2]])
STAT_STD = np.maximum(STAT_STD, 1e-3)


def stats_maps(codes, rows=128):
    out = np.empty((N_STATS, *codes.shape[-2:]), dtype=np.float16)
    for r in range(0, codes.shape[-2], rows):
        out[:, r:r + rows] = standardise(stats_raw(codes[:, :, r:r + rows])[None], STAT_MEAN, STAT_STD)[0]
    return out


t0 = time.time()
STATIC_TILE = {site: stats_maps(x) for site, x in TILE_X.items()}
STATIC_X = {s: np.stack([stats_maps(x) for x in X[s]]) for s in ("val", "test")}
gb = (sum(m.nbytes for m in STATIC_TILE.values()) + sum(m.nbytes for m in STATIC_X.values())) / 1e9
print(f"{N_STATS} seasonal statistics precomputed in {time.time() - t0:.0f}s ({gb:.1f} GB)")

fig, axes = plt.subplots(3, len(examples), figsize=(4 * len(examples), 12), squeeze=False)
for j, i in enumerate(examples):
    site, r, c = train_window(i)
    s = STATIC_TILE[site][:, r:r + PATCH, c:c + PATCH].astype(np.float32)
    for row, (name, cmap) in enumerate([("VH mean", "viridis"), ("VH std", "magma"), ("water months", "Blues")]):
        axes[row, j].imshow(s[STAT_NAMES.index(name)], cmap=cmap, vmin=-2, vmax=3)
        axes[row, j].set_title(f"{site}: {name}" if row == 0 else name)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

**How to read this.** *VH mean* is bright for forest and buildings, dark for water and bare soil. *VH std* is bright where vegetation changes over the year (fields) and dark for forest, cities and water. *Water months* is dark blue for permanent water and lighter for land that is flooded only in some months.

## 5. Datasets, augmentation and class balancing

Unchanged from the 1st iteration (section 6 there): one item = one patch, one random target month, the annual label; random crop position (±128 px), flips and 90° rotations, VV / VH brightness shift (±1 dB) and 0.5 dB noise; class weights `sqrt(median_freq / freq)` and a class-aware patch sampler; batches cut on the GPU when the data fits.

Changed:
- **Month dropout 0.3.** Each input month is blanked with chance `MONTH_DROP` (for `month_year` that is the whole monthly input). The model then has only the yearly statistics and must still get the class right, which is exactly what stops a dark winter month from turning forest into water.
- **No PCA or terrain channels**: the static input is the 9 seasonal statistics for every variant.
- `year_stats` has no monthly input at all (`take_months` with an empty window).
- Class weights and the patch sampler count every WorldCover pixel.
Final iteration: class weights come from the reference windows (as in `Iter2_Test_1_Full`); `PATCH_WEIGHTS` is 0 for windows outside `TRAIN_YEARS` / `USE_NEW_SITES`, so only the chosen data is drawn. Each epoch still draws `SAMPLES_PER_EPOCH` windows.


In [ ]:
def window_codes(codes, month, variant):
    return take_months(codes, window_months(month, variant["n_months"]))


LUT_T = torch.tensor(DB_LUT, device=DEVICE)
CH_MEAN_T = torch.tensor(CH_MEAN, device=DEVICE).view(1, 1, 3, 1, 1)
CH_STD_T = torch.tensor(CH_STD, device=DEVICE).view(1, 1, 3, 1, 1)


def gpu_input(codes, static, shift_db=None, month_drop=0.0):
    """uint8 codes (B, months, 2, H, W) + float16 statistics (B, S, H, W) -> standardised model input on the GPU.

    VV, VH, VV-VH per month (oldest first), then the seasonal statistics. shift_db (B, 2) adds the VV / VH
    brightness shift and 0.5 dB noise; month_drop blanks each month with this chance (code 0 -> 0 after
    standardisation).
    """
    static = static.to(DEVICE, non_blocking=True).float()
    if codes.shape[1] == 0:
        return static.contiguous(memory_format=torch.channels_last)
    codes = codes.to(DEVICE, non_blocking=True).long()
    if month_drop > 0:
        drop = torch.rand(codes.shape[:2], device=DEVICE) < month_drop
        codes = codes.masked_fill(drop[:, :, None, None, None], 0)
    db = LUT_T[codes]
    if shift_db is not None:
        db = db + shift_db.view(-1, 1, 2, 1, 1) + 0.5 * torch.randn_like(db)
    vv, vh = db[:, :, 0], db[:, :, 1]
    f = torch.nan_to_num((torch.stack([vv, vh, vv - vh], 2) - CH_MEAN_T) / CH_STD_T, nan=0.0).flatten(1, 2)
    return torch.cat([f, static], 1).contiguous(memory_format=torch.channels_last)


def gpu_augment(x, y):
    """Random flips and 90-degree rotations, drawn per sample."""
    for i in range(len(x)):
        dims = [d for d, flip in zip((-1, -2), torch.rand(2) < 0.5) if flip]
        xi, yi = (x[i].flip(dims), y[i].flip(dims)) if dims else (x[i], y[i])
        k = int(torch.randint(4, ()))
        x[i], y[i] = torch.rot90(xi, k, (-2, -1)), torch.rot90(yi, k, (-2, -1))
    return x, y


def input_channels(variant):
    return 3 * variant["n_months"] + N_STATS


class TrainDataset(Dataset):
    def __init__(self, variant):
        self.variant = variant

    def __len__(self):
        return len(TRAIN_ORIGINS)

    def __getitem__(self, i):
        site, r, c = train_window(i, *np.random.randint(-CROP_JITTER, CROP_JITTER + 1, 2))
        month = np.random.randint(12)
        codes = window_codes(TILE_X[site][:, :, r:r + PATCH, c:c + PATCH], month, self.variant)
        static = STATIC_TILE[site][:, r:r + PATCH, c:c + PATCH]
        label = TILE_Y[site][r:r + PATCH, c:c + PATCH].astype(np.int64)
        return torch.from_numpy(codes), torch.from_numpy(static), torch.from_numpy(label), month


class EvalDataset(Dataset):
    def __init__(self, x, y, static, variant, months=range(12)):
        self.x, self.y, self.static, self.variant = x, y, static, variant
        self.items = [(i, m) for i in range(len(x)) for m in months]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        i, month = self.items[k]
        codes = window_codes(self.x[i], month, self.variant)
        return (torch.from_numpy(codes), torch.from_numpy(self.static[i]),
                torch.from_numpy(self.y[i].astype(np.int64)), month)


def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2 ** 32)


def size_gb(arrays):
    return sum(a.nbytes for a in arrays) / 1e9


EVAL_SPLITS = ("val", "test")
train_gb = size_gb([*TILE_X.values(), *STATIC_TILE.values(), *TILE_Y.values()])
eval_gb = size_gb([X[s] for s in EVAL_SPLITS] + [STATIC_X[s] for s in EVAL_SPLITS] + [Y[s] for s in EVAL_SPLITS])
free_gb = 0.9 * GPU_MEM_GB - GPU_TRAIN_RESERVE_GB
EVAL_ON_GPU = DEVICE.type == "cuda" and eval_gb <= free_gb
TRAIN_ON_GPU = EVAL_ON_GPU and eval_gb + train_gb <= free_gb

if EVAL_ON_GPU:
    X_GPU = {s: torch.from_numpy(X[s]).to(DEVICE) for s in EVAL_SPLITS}
    STATIC_GPU = {s: torch.from_numpy(STATIC_X[s]).to(DEVICE) for s in EVAL_SPLITS}
    Y_GPU = {s: torch.from_numpy(Y[s]).to(DEVICE) for s in EVAL_SPLITS}

TRAIN_SITE_ID = {site: k for k, site in enumerate(TILE_X)}
if TRAIN_ON_GPU:
    size = max(TILE_Y[site].shape[0] for site in TILE_X)
    TILE_X_GPU = torch.zeros((len(TILE_X), 12, 2, size, size), dtype=torch.uint8, device=DEVICE)
    TILE_S_GPU = torch.zeros((len(TILE_X), N_STATS, size, size), dtype=torch.float16, device=DEVICE)
    TILE_Y_GPU = torch.full((len(TILE_X), size, size), IGNORE, dtype=torch.int16, device=DEVICE)
    for site, k in TRAIN_SITE_ID.items():
        h, w = TILE_Y[site].shape
        TILE_X_GPU[k, :, :, :h, :w] = torch.from_numpy(TILE_X[site])
        TILE_S_GPU[k, :, :h, :w] = torch.from_numpy(STATIC_TILE[site])
        TILE_Y_GPU[k, :h, :w] = torch.from_numpy(TILE_Y[site])

print(f"data: train {train_gb:.1f} GB, val + test {eval_gb:.1f} GB | GPU space for data: {max(free_gb, 0):.1f} GB")
print("train batches:", "cut on the GPU" if TRAIN_ON_GPU else f"DataLoader, {NUM_WORKERS} workers",
      "| val/test batches:", "cut on the GPU" if EVAL_ON_GPU else f"DataLoader, {NUM_WORKERS} workers")


def gpu_train_batches(variant, steps):
    """Same sampling as TrainDataset + WeightedRandomSampler, but cut from the tiles on the GPU."""
    ids = torch.multinomial(torch.as_tensor(PATCH_WEIGHTS, dtype=torch.double), steps * BATCH, replacement=True)
    for b in range(steps):
        codes, static, labels = [], [], []
        for i in ids[b * BATCH:(b + 1) * BATCH].tolist():
            site, r, c = train_window(i, *np.random.randint(-CROP_JITTER, CROP_JITTER + 1, 2))
            k = TRAIN_SITE_ID[site]
            months = window_months(np.random.randint(12), variant["n_months"])
            codes.append(take_months(TILE_X_GPU[k, :, :, r:r + PATCH, c:c + PATCH], months))
            static.append(TILE_S_GPU[k, :, r:r + PATCH, c:c + PATCH])
            labels.append(TILE_Y_GPU[k, r:r + PATCH, c:c + PATCH])
        yield torch.stack(codes), torch.stack(static), torch.stack(labels).long()


def eval_batches(split, variant, months=range(12), ids=None):
    """Fixed val/test patches, each with every month in `months`: (codes, static, label, month) batches."""
    ids = range(len(X[split])) if ids is None else [int(i) for i in ids]
    if not EVAL_ON_GPU:
        sel = slice(None) if isinstance(ids, range) else ids
        ds = EvalDataset(X[split][sel], Y[split][sel], STATIC_X[split][sel], variant, months)
        yield from DataLoader(ds, 2 * BATCH, num_workers=NUM_WORKERS, pin_memory=True)
        return
    items = [(i, m) for i in ids for m in months]
    for b in range(0, len(items), 2 * BATCH):
        chunk = items[b:b + 2 * BATCH]
        codes = torch.stack([take_months(X_GPU[split][i], window_months(m, variant["n_months"])) for i, m in chunk])
        static = torch.stack([STATIC_GPU[split][i] for i, _ in chunk])
        labels = torch.stack([Y_GPU[split][i] for i, _ in chunk]).long()
        yield codes, static, labels, torch.tensor([m for _, m in chunk], device=DEVICE)


freq = counts["train_ref"] / counts["train_ref"].sum()
CLASS_WEIGHTS = np.sqrt(np.median(freq) / np.clip(freq, 1e-6, None))
CLASS_WEIGHTS = (CLASS_WEIGHTS / CLASS_WEIGHTS.mean()).astype(np.float32)
display(pd.DataFrame({"train share %": (freq * 100).round(2), "loss weight": CLASS_WEIGHTS.round(2)},
                     index=CLASS_NAMES))

patch_classes = label_classes(Y["train"])
patch_frac = np.stack([np.bincount(t[t >= 0], minlength=N_CLASSES) / max((t >= 0).sum(), 1) for t in patch_classes])
PATCH_WEIGHTS = np.where(patch_frac >= 0.01, CLASS_WEIGHTS[None], 0).max(1)
# only the windows of TRAIN_YEARS / USE_NEW_SITES; a window seen in several years shares its weight between
# them, so the later years add variety but do not change the class balance
WINDOW_KEY = (train_all.site + "_" + train_all.row.astype(str) + "_" + train_all.col.astype(str)).to_numpy()
n_years = pd.Series(USED.astype(int)).groupby(WINDOW_KEY).transform("sum").to_numpy()
PATCH_WEIGHTS = np.where(USED, PATCH_WEIGHTS / np.maximum(n_years, 1), 0.0)
ref_w = np.where(REF, np.where(patch_frac >= 0.01, CLASS_WEIGHTS[None], 0).max(1), 0.0)
display(pd.DataFrame({"reference (Iter2_Test_1_Full) %": (ref_w / ref_w.sum()) @ patch_frac * 100,
                      "this run %": (PATCH_WEIGHTS / PATCH_WEIGHTS.sum()) @ patch_frac * 100},
                     index=CLASS_NAMES).round(1).rename_axis("expected share of drawn pixels"))
print("patch sampling weight range:", PATCH_WEIGHTS.min().round(2), "-", PATCH_WEIGHTS.max().round(2))

## 6. The CNN model: U-Net with a ResNet-34 encoder

Unchanged (1st iteration, section 7): ImageNet-pretrained ResNet-34 encoder with the first layer adapted to the channel count, U-Net decoder with skip connections, dropout before the 1 × 1 classifier. Input channels: `year_stats` 9, `month_year` 12, `three_month_year` 18.

The skip connection from the full-resolution stem keeps the 10 m detail. In the 1st iteration that detail was lost before the network ever saw it (median filter) and never trained at the boundaries (consensus mask); both are fixed in the data now.

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], 1)
        return self.conv(x)


class UNetResNet34(nn.Module):
    def __init__(self, in_ch, n_classes, pretrained=True):
        super().__init__()
        try:
            enc = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)
        except Exception as err:
            print("pretrained weights unavailable, using random init:", err)
            enc = torchvision.models.resnet34(weights=None)
        conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            conv1.weight.copy_(enc.conv1.weight.mean(1, keepdim=True).repeat(1, in_ch, 1, 1) * 3 / in_ch)
        self.stem = nn.Sequential(conv1, enc.bn1, enc.relu)
        self.pool = enc.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = enc.layer1, enc.layer2, enc.layer3, enc.layer4
        self.dec4 = UpBlock(512, 256, 256)
        self.dec3 = UpBlock(256, 128, 128)
        self.dec2 = UpBlock(128, 64, 64)
        self.dec1 = UpBlock(64, 64, 64)
        self.dec0 = UpBlock(64, 0, 32)
        self.dropout = nn.Dropout2d(DROPOUT)
        self.head = nn.Conv2d(32, n_classes, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.enc1(self.pool(s0))
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        d = self.dec4(s4, s3)
        d = self.dec3(d, s2)
        d = self.dec2(d, s1)
        d = self.dec1(d, s0)
        d = self.dec0(d)
        return self.head(self.dropout(d))


for name, variant in ALL_VARIANTS.items():
    print(f"{name:17} input channels: {input_channels(variant)}")
m = UNetResNet34(3, N_CLASSES, pretrained=False)
print(f"parameters: {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M")
print("output shape:", tuple(m(torch.zeros(1, 3, PATCH, PATCH)).shape))
del m

## 7. Loss and metrics

**Loss = weighted cross-entropy on every WorldCover pixel + 0.5 × Lovász-softmax on agreement pixels**

| Part | Pixels | Why |
|---|---|---|
| cross-entropy (class weights, label smoothing 0.1) | all WorldCover pixels; agreement pixels weight 1, disagreement pixels `DISAGREE_WEIGHT` = 0.5 | boundaries (forest edges, field edges, roads, streams) are mostly disagreement pixels. The 1st iteration ignored them, so the model never learned where an edge is. Weight 0.5 still trains them, but trusts them less, because there one of the two maps is wrong |
| Lovász-softmax (1st iteration, section 8) | agreement pixels only | it optimises IoU directly; on uncertain pixels it would chase label noise |

The **logged loss** stays the plain weighted cross-entropy on agreement pixels, so the curves read the same as in the 1st iteration.

**Metrics.** OA, IoU, mIoU and F1 as before. Every score is computed twice:
- on **agreement pixels** (main score; the same pixels the 1st iteration was scored on);
- on **all WorldCover pixels** (`_all`), which includes boundaries. Lower, because one of the two maps is wrong on many of these pixels, but it shows whether edges improve.

In [ ]:
def lovasz_grad(fg_sorted):
    total = fg_sorted.sum()
    intersection = total - fg_sorted.cumsum(0)
    union = total + (1 - fg_sorted).cumsum(0)
    jaccard = 1 - intersection / union
    jaccard[1:] = jaccard[1:] - jaccard[:-1]
    return jaccard


def lovasz_softmax(probs, target, sample=1.0):
    probs = probs.permute(0, 2, 3, 1).reshape(-1, probs.shape[1])
    target = target.reshape(-1)
    valid = (target != IGNORE) & (torch.rand(len(target), device=target.device) < sample)
    probs, target = probs[valid], target[valid]
    losses = []
    for c in range(probs.shape[1]):
        fg = (target == c).float()
        if fg.sum() == 0:
            continue
        errors, order = (fg - probs[:, c]).abs().sort(descending=True)
        losses.append(torch.dot(errors, lovasz_grad(fg[order])))
    return torch.stack(losses).mean() if losses else probs.sum() * 0


class WorldCoverLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.w = class_weights

    def agree_ce(self, logits, y):
        """Plain weighted cross-entropy on agreement pixels: the logged loss."""
        target, agree = split_label(y)
        return F.cross_entropy(logits.float(), target.masked_fill(~agree, IGNORE), weight=self.w, ignore_index=IGNORE)

    def forward(self, logits, y):
        """Returns (total loss to optimise, logged cross-entropy)."""
        logits = logits.float()
        target, agree = split_label(y)
        pixel_w = (target != IGNORE).float() * (DISAGREE_WEIGHT + (1 - DISAGREE_WEIGHT) * agree.float())
        ce = F.cross_entropy(logits, target, weight=self.w, ignore_index=IGNORE, reduction="none",
                             label_smoothing=LABEL_SMOOTHING)
        ce = (ce * pixel_w).sum() / (self.w[target.clamp(min=0)] * pixel_w).sum().clamp(min=1e-6)
        target_agree = target.masked_fill(~agree, IGNORE)
        lovasz = lovasz_softmax(logits.softmax(1), target_agree, LOVASZ_SAMPLE)
        logged = F.cross_entropy(logits, target_agree, weight=self.w, ignore_index=IGNORE)
        return ce + LOVASZ_WEIGHT * lovasz, logged.detach()


def update_confusion(conf, pred, target):
    valid = target != IGNORE
    conf += torch.bincount(target[valid] * N_CLASSES + pred[valid],
                           minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
    return conf


def metrics_from_confusion(conf):
    conf = conf.double()
    tp = conf.diag()
    fp, fn = conf.sum(0) - tp, conf.sum(1) - tp
    iou = tp / (tp + fp + fn).clamp(min=1)
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)
    present = conf.sum(1) > 0
    return {"OA": (tp.sum() / conf.sum().clamp(min=1)).item(),
            "mIoU": iou[present].mean().item(),
            "IoU": iou.cpu().numpy(), "F1": f1.cpu().numpy(),
            "precision": (tp / (tp + fp).clamp(min=1)).cpu().numpy(),
            "recall": (tp / (tp + fn).clamp(min=1)).cpu().numpy()}


criterion = WorldCoverLoss(torch.tensor(CLASS_WEIGHTS, device=DEVICE))

## 8. Training

Unchanged from the 1st iteration (section 9 explains every part and how to read the log): AdamW (lr 6e-4, weight decay 0.05 on convolution weights), mixed precision, 1 warmup epoch + cosine schedule, EMA weights for validation, saving and test, early stopping on val mIoU (agreement pixels, all 12 months), best weights saved at every improvement.

The log line is the same:

```
epoch  7 | train loss 0.929 mIoU 0.635 | val loss 1.026 OA 0.802 mIoU 0.585 all 0.512 | lr 2.6e-04 | 41s *
```

New: `all` is the val mIoU on all WorldCover pixels (including boundaries). Model choice still uses the agreement mIoU.

In [ ]:
def evaluate(model, batches):
    """Scores on agreement pixels (main) and on all WorldCover pixels (_all), one prediction per (patch, month)."""
    model.eval()
    conf = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE)
    conf_all = torch.zeros_like(conf)
    loss_sum, n = 0.0, 0
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        for codes, static, yb, _ in batches:
            xb, yb = gpu_input(codes, static), yb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            loss_sum += criterion.agree_ce(logits, yb).item() * len(xb)
            n += len(xb)
            pred = logits.argmax(1)
            target, agree = split_label(yb)
            update_confusion(conf, pred, target.masked_fill(~agree, IGNORE))
            update_confusion(conf_all, pred, target)
    out = metrics_from_confusion(conf)
    out["mIoU_all"] = metrics_from_confusion(conf_all)["mIoU"]
    out["loss"], out["confusion"] = loss_sum / n, conf.cpu().numpy()
    return out


def train_variant(name, variant):
    print(f"=== {name}: {input_channels(variant)} input channels ===")
    steps = SAMPLES_PER_EPOCH // BATCH
    if not TRAIN_ON_GPU:
        sampler = WeightedRandomSampler(torch.as_tensor(PATCH_WEIGHTS, dtype=torch.double),
                                        num_samples=SAMPLES_PER_EPOCH, replacement=True)
        train_dl = DataLoader(TrainDataset(variant), BATCH, sampler=sampler, num_workers=NUM_WORKERS,
                              pin_memory=True, drop_last=True, persistent_workers=NUM_WORKERS > 0,
                              worker_init_fn=seed_worker)

    model = UNetResNet34(input_channels(variant), N_CLASSES).to(DEVICE, memory_format=torch.channels_last)
    ema = torch.optim.swa_utils.AveragedModel(
        model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(EMA_DECAY), use_buffers=True)
    decay = [p for p in model.parameters() if p.ndim > 1]
    no_decay = [p for p in model.parameters() if p.ndim <= 1]
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                                   {"params": no_decay, "weight_decay": 0.0}], lr=LEARNING_RATE)
    warmup, total = WARMUP_EPOCHS * steps, MAX_EPOCHS * steps

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * min(step - warmup, total - warmup) / (total - warmup)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler(enabled=USE_SCALER)

    history, best, best_state, wait = [], -1.0, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0, loss_sum, n = time.time(), 0.0, 0
        conf = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE)
        batches = (gpu_train_batches(variant, steps) if TRAIN_ON_GPU
                   else ((codes, static, yb) for codes, static, yb, _ in train_dl))
        for codes, static, yb in batches:
            yb = yb.to(DEVICE, non_blocking=True)
            xb = gpu_input(codes, static, shift_db=torch.empty(len(yb), 2, device=DEVICE).uniform_(-1, 1),
                           month_drop=MONTH_DROP)
            xb, yb = gpu_augment(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(xb)
                loss, ce = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema.update_parameters(model)
            loss_sum += ce.item() * len(xb)
            n += len(xb)
            target, agree = split_label(yb)
            update_confusion(conf, logits.detach().argmax(1), target.masked_fill(~agree, IGNORE))
        train_m = metrics_from_confusion(conf)
        validate = epoch % VAL_EVERY == 0 or epoch == MAX_EPOCHS
        val_m = (evaluate(ema.module, eval_batches("val", variant, VAL_MONTHS)) if validate
                 else {"loss": np.nan, "OA": np.nan, "mIoU": np.nan, "mIoU_all": np.nan})
        history.append({"epoch": epoch, "train_loss": loss_sum / n, "val_loss": val_m["loss"],
                        "train_OA": train_m["OA"], "val_OA": val_m["OA"],
                        "train_mIoU": train_m["mIoU"], "val_mIoU": val_m["mIoU"], "val_mIoU_all": val_m["mIoU_all"],
                        "lr": optimizer.param_groups[0]["lr"]})
        improved = validate and val_m["mIoU"] > best
        if improved:
            best, wait = val_m["mIoU"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
            torch.save(best_state, VARIANT_DIR / f"{name}.pt")
        elif validate:
            wait += VAL_EVERY
        print(f"epoch {epoch:2d} | train loss {loss_sum / n:.3f} mIoU {train_m['mIoU']:.3f} | "
              f"val loss {val_m['loss']:.3f} OA {val_m['OA']:.3f} mIoU {val_m['mIoU']:.3f} all {val_m['mIoU_all']:.3f} | "
              f"lr {optimizer.param_groups[0]['lr']:.1e} | {time.time() - t0:.0f}s{' *' if improved else ''}")
        if wait >= PATIENCE:
            print(f"early stopping: no val mIoU improvement for {PATIENCE} epochs")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, VARIANT_DIR / f"{name}.pt")
    pd.DataFrame(history).to_csv(VARIANT_DIR / f"{name}_history.csv", index=False)
    print(f"best val mIoU {best:.3f}, weights saved to {VARIANT_DIR / f'{name}.pt'}")
    return model, pd.DataFrame(history)

In [ ]:
RUN_START = time.time()
models, histories, minutes = {}, {}, {}
for name, variant in VARIANTS.items():
    t0 = time.time()
    models[name], histories[name] = train_variant(name, variant)
    minutes[name] = round((time.time() - t0) / 60, 1)
json.dump({"minutes_per_variant": minutes, "total_minutes": round((time.time() - RUN_START) / 60, 1)},
          open(OUT / "timing.json", "w"), indent=2)
print("minutes per variant:", minutes)

## 9. Results

### 9.1 Training curves

How to read them: 1st iteration, section 10.1. The dashed line marks the best epoch (its weights are kept). The val mIoU on all WorldCover pixels is drawn as a dotted line in the mIoU panel.

In [ ]:
fig, axes = plt.subplots(len(histories), 3, figsize=(18, 4.5 * len(histories)), squeeze=False)
for row, (name, h) in enumerate(histories.items()):
    best_epoch = h.epoch[h.val_mIoU.idxmax()]
    v = h.dropna(subset=["val_mIoU"])
    for ax, metric in zip(axes[row], ["loss", "OA", "mIoU"]):
        ax.plot(h.epoch, h[f"train_{metric}"], label="train")
        ax.plot(v.epoch, v[f"val_{metric}"], marker="o", label="val")
        if metric == "mIoU":
            ax.plot(v.epoch, v.val_mIoU_all, ls=":", marker=".", label="val, all WorldCover pixels")
        ax.axvline(best_epoch, color="grey", ls="--", lw=1, label="best epoch")
        ax.set_title(f"{name}: {metric}")
        ax.set_xlabel("epoch")
        ax.legend()
    gap = h.train_mIoU[h.val_mIoU.idxmax()] - h.val_mIoU.max()
    print(f"{name}: best epoch {best_epoch}, train-val mIoU gap at best epoch {gap:.3f} (1st iteration: 0.18-0.21)")
plt.tight_layout()
plt.savefig(OUT / "training_curves.png", dpi=120)
plt.show()

### 9.2 Test scores: rural sites, temporal smoothing and flicker

Every test patch is predicted for **all 12 months**. Then:

- **raw**: each month's map on its own (what the 1st iteration reported);
- **smoothed**: the class probabilities of months t-1, t and t+1 are averaged before the class is chosen (the window wraps around the year, as the model's month window does). This is cheap to do in the prototype as well;
- **flicker**: the share of pixels whose class changes between consecutive months. The labels do not change within 2021, so for most pixels any change is a false change. `year_stats` has flicker 0 by construction.

Columns: `mIoU` on agreement pixels of all test sites, `rural mIoU` on agreement pixels of the non-town test sites (**the main score**), `rural mIoU all` on all WorldCover pixels of the rural sites (boundaries included).

`BEST` (the model saved as `final_model.pt`) is `FINAL_VARIANT` in the final run; in DEV it is the variant with the best **val** mIoU among those with monthly maps. Test is only reported. The prototype should use smoothing if it scores at least as well as raw.

In [ ]:
def predict_year(model, split, variant, ids):
    """Softmax for all 12 months of the given patches: (len(ids), 12, classes, H, W) on the device."""
    batches = list(eval_batches(split, variant, range(12), ids))
    codes = torch.cat([b[0] for b in batches])
    static = torch.cat([b[1] for b in batches])
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        logits = model(gpu_input(codes, static))
    return logits.float().softmax(1).view(len(ids), 12, N_CLASSES, *logits.shape[-2:])


def smooth_months(probs):
    """Mean probability of months t-1, t, t+1 (wrapping around the year)."""
    return (probs.roll(1, 1) + probs + probs.roll(-1, 1)) / 3


def evaluate_year(model, split, variant, ids=None):
    model.eval()
    ids = list(range(len(X[split]))) if ids is None else [int(i) for i in ids]
    chunk_size = max(1, 2 * BATCH // 12)
    conf = {k: torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE)
            for k in ("raw", "smooth", "raw_all", "smooth_all")}
    month_conf = torch.zeros(12, N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE)
    flicker = {"raw": 0.0, "smooth": 0.0}
    for b in range(0, len(ids), chunk_size):
        chunk = ids[b:b + chunk_size]
        probs = predict_year(model, split, variant, chunk)
        y = torch.as_tensor(Y[split][chunk], device=DEVICE).long()[:, None].expand(-1, 12, -1, -1)
        target, agree = split_label(y)
        target_agree = target.masked_fill(~agree, IGNORE)
        for key, p in (("raw", probs), ("smooth", smooth_months(probs))):
            pred = p.argmax(2)
            update_confusion(conf[key], pred, target_agree)
            update_confusion(conf[f"{key}_all"], pred, target)
            flicker[key] += (pred[:, 1:] != pred[:, :-1]).float().mean().item() * len(chunk)
            if key == "raw":
                for m in range(12):
                    update_confusion(month_conf[m], pred[:, m], target_agree[:, m])
    out = {k: metrics_from_confusion(c) for k, c in conf.items()}
    out["flicker"] = {k: v / len(ids) for k, v in flicker.items()}
    out["month_mIoU"] = [metrics_from_confusion(month_conf[m])["mIoU"] for m in range(12)]
    out["confusion"] = conf["raw"].cpu().numpy()
    return out


RURAL_IDS = index["test"].index[index["test"].focus != "urban"].to_numpy()
results, results_rural = {}, {}
print(f"12-month test: {len(X['test'])} patches x 12 months, all sites and then the {len(RURAL_IDS)} rural patches")
for name, variant in VARIANTS.items():
    t0 = time.time()
    results[name] = evaluate_year(models[name], "test", variant)
    print(f"{name}: all sites mIoU {results[name]['raw']['mIoU']:.3f} | {time.time() - t0:.0f}s", flush=True)
    t0 = time.time()
    results_rural[name] = evaluate_year(models[name], "test", variant, RURAL_IDS)
    print(f"{name}: rural mIoU {results_rural[name]['raw']['mIoU']:.3f} | {time.time() - t0:.0f}s", flush=True)

rows = {}
for name in VARIANTS:
    r, rr = results[name], results_rural[name]
    rows[name] = {"best val mIoU": histories[name].val_mIoU.max(), "epochs": len(histories[name]),
                  "OA": r["raw"]["OA"], "mIoU": r["raw"]["mIoU"], "mIoU smoothed": r["smooth"]["mIoU"],
                  "rural mIoU": rr["raw"]["mIoU"], "rural mIoU smoothed": rr["smooth"]["mIoU"],
                  "rural mIoU all": rr["raw_all"]["mIoU"], "rural mIoU all smoothed": rr["smooth_all"]["mIoU"],
                  "flicker %": rr["flicker"]["raw"] * 100, "flicker smoothed %": rr["flicker"]["smooth"] * 100}
comparison = pd.DataFrame(rows).T
display(comparison.round(4))

per_class = pd.concat({name: pd.DataFrame({"IoU": r["raw"]["IoU"], "IoU smoothed": r["smooth"]["IoU"],
                                           "precision": r["raw"]["precision"], "recall": r["raw"]["recall"]},
                                          index=CLASS_NAMES)
                       for name, r in results_rural.items()}, axis=1)
print("per-class scores on the rural test sites (agreement pixels):")
display(per_class.round(3))

monthly = [name for name in VARIANTS if VARIANTS[name]["n_months"] > 0] or list(VARIANTS)
BEST = comparison.loc[monthly, "best val mIoU"].idxmax()
USE_SMOOTHING = results_rural[BEST]["smooth"]["mIoU"] >= results_rural[BEST]["raw"]["mIoU"]
print("final model (chosen by val mIoU among the trained variants, test is only reported):", BEST)
print("temporal smoothing for the prototype:", "yes" if USE_SMOOTHING else "no")

json.dump({name: {split_name: {key: {"OA": res[key]["OA"], "mIoU": res[key]["mIoU"],
                                     "IoU": dict(zip(CLASS_NAMES, res[key]["IoU"].round(4).tolist())),
                                     "F1": dict(zip(CLASS_NAMES, res[key]["F1"].round(4).tolist()))}
                               for key in ("raw", "smooth", "raw_all", "smooth_all")}
                  | {"flicker": res["flicker"], "month_mIoU": [round(v, 4) for v in res["month_mIoU"]]}
                  for split_name, res in (("all_test_sites", results[name]), ("rural_test_sites", results_rural[name]))}
           for name in VARIANTS}, open(OUT / "test_metrics.json", "w"), indent=2)

### 9.3 Confusion matrices (rural test sites)

Rows are true classes, columns predicted classes. The first matrix is row-normalised (recall on the diagonal). Raw monthly maps of `BEST`, agreement pixels of the rural test sites. Watch the Trees row: in the 1st iteration, forest in winter leaked into water, wetland and cropland.

The second matrix is **column**-normalised: the diagonal is the **precision** of each class (how many of its predictions are right). High recall with low precision means the model **over-predicts** that class, for example because its class weight is too high; then lower that weight rather than add data.

In [ ]:
conf = results_rural[BEST]["confusion"].astype(np.float64)
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
for ax, axis, title in [(axes[0], 1, "row-normalised: recall"), (axes[1], 0, "column-normalised: precision")]:
    norm = conf / conf.sum(axis, keepdims=True).clip(min=1)
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center",
                    color="white" if norm[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=40, ha="right")
    ax.set_yticks(range(N_CLASSES), CLASS_NAMES)
    ax.set_xlabel("predicted")
    ax.set_ylabel("reference")
    ax.set_title(f"Rural test sites, {title} ({BEST})")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=120)
plt.show()

### 9.4 Scores per region, site type and month

- **Per site type**: `mosaic` is the mixed rural landscape the product is for; read `flicker %` next to mIoU.
- **Per month** (raw maps, rural sites): the labels are the same all year, so a drop in some months is the **seasonal effect** (snow, frozen or wet soil, harvested fields). With the yearly statistics and month dropout this curve should be much flatter than in the 1st iteration (January mIoU there: lowest of all months).

In [ ]:
def group_scores(model, variant, column):
    rows = []
    t0 = time.time()
    for group, df in index["test"].groupby(column):
        r = evaluate_year(model, "test", variant, df.index.to_numpy())
        print(f"{column} {group}: {len(df)} patches, mIoU {r['raw']['mIoU']:.3f} | {time.time() - t0:.0f}s", flush=True)
        rows.append({column: group, "patches": len(df), "OA": r["raw"]["OA"], "mIoU": r["raw"]["mIoU"],
                     "mIoU smoothed": r["smooth"]["mIoU"], "flicker %": r["flicker"]["raw"] * 100,
                     "flicker smoothed %": r["flicker"]["smooth"] * 100})
    return pd.DataFrame(rows).set_index(column).round(3)


display(group_scores(models[BEST], VARIANTS[BEST], "region"))
display(group_scores(models[BEST], VARIANTS[BEST], "focus"))

plt.figure(figsize=(10, 4))
for name, r in results_rural.items():
    plt.plot(MONTH_NAMES, r["month_mIoU"], marker="o", label=name)
plt.ylabel("test mIoU, rural sites")
plt.title("Rural test mIoU per month (seasonal effect, raw monthly maps)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "month_miou.png", dpi=120)
plt.show()

### 9.5 Twelve monthly maps of unseen rural mosaics

The same view as the prototype: one test patch per mosaic test site (including `black_forest_villages`), all 12 months. Row 1: radar, row 2: raw monthly maps, row 3: smoothed maps. The last column shows the WorldCover label and the agreement mask. A good result: the same classes in every month, sharp field and forest edges, no winter water.

In [ ]:
model = models[BEST].eval()
mosaic_test = index["test"][index["test"].focus == "mosaic"]
picks = mosaic_test.groupby("site").head(1).index.tolist()[:4]
for i in picks:
    probs = predict_year(model, "test", VARIANTS[BEST], [i])
    raw = probs.argmax(2)[0].cpu().numpy()
    smooth = smooth_months(probs).argmax(2)[0].cpu().numpy()
    fig, axes = plt.subplots(3, 13, figsize=(26, 6.6))
    for m in range(12):
        axes[0, m].imshow(false_colour(X["test"][i, m]))
        axes[0, m].set_title(MONTH_NAMES[m])
        axes[1, m].imshow(raw[m] + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
        axes[2, m].imshow(smooth[m] + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
    axes[0, 12].imshow(label_classes(Y["test"][i]) + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
    axes[0, 12].set_title("WorldCover")
    axes[1, 12].imshow((Y["test"][i] >= 0) & (Y["test"][i] < N_CLASSES), cmap="gray", vmin=0, vmax=1,
                       interpolation="nearest")
    axes[1, 12].set_title("agreement")
    axes[1, 0].set_ylabel("raw")
    axes[2, 0].set_ylabel("smoothed")
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    axes[2, 12].axis("off")
    fig.suptitle(f"{index['test'].site[i]} ({BEST})")
    fig.legend(handles=LEGEND, loc="lower center", ncol=9)
    plt.tight_layout(rect=(0, 0.05, 1, 0.97))
    plt.savefig(OUT / f"months_{index['test'].site[i]}.png", dpi=100)
    plt.show()

### 9.6 Class bias for Water and Man-made

Water is over-predicted (Iter2_Test_1_Full: test precision 0.59, recall 0.74) because dark, smooth radar returns from radar shadow, wet soil or dark forest look like open water; bright bare rock can look like Man-made. A per-class **offset on the log-probabilities** shifts these decisions without retraining.

- Tuned on **val** (all 12 months, with the smoothing of the model card), one class after the other (`BIAS_CLASSES`) over `BIAS_GRID`; an offset is kept only if val mIoU rises by more than `BIAS_MIN_GAIN`.
- The test set only reports the effect (table: no bias vs with bias).
- Stored as `class_logit_bias` in the model card; the prototype adds it before choosing the class.


In [ ]:
def bias_confusions(model, split, ids, biases, smooth):
    """Metrics (agreement pixels) per candidate bias vector, one pass over the patches."""
    model.eval()
    confs = [torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE) for _ in biases]
    chunk_size = max(1, 2 * BATCH // 12)
    ids = [int(i) for i in ids]
    for b in range(0, len(ids), chunk_size):
        chunk = ids[b:b + chunk_size]
        probs = predict_year(model, split, VARIANTS[BEST], chunk)
        if smooth:
            probs = smooth_months(probs)
        logp = probs.clamp_min(1e-8).log()
        y = torch.as_tensor(Y[split][chunk], device=DEVICE).long()[:, None].expand(-1, 12, -1, -1)
        target, agree = split_label(y)
        target_agree = target.masked_fill(~agree, IGNORE)
        for conf, bias in zip(confs, biases):
            update_confusion(conf, (logp + bias.view(1, 1, -1, 1, 1)).argmax(2), target_agree)
    return [metrics_from_confusion(c) for c in confs]


bias = torch.zeros(N_CLASSES, device=DEVICE)
val_ids = range(len(X["val"]))
best_val = start_val = bias_confusions(models[BEST], "val", val_ids, [bias], USE_SMOOTHING)[0]["mIoU"]
for name in BIAS_CLASSES:
    k = CLASS_NAMES.index(name)
    candidates = []
    for v in BIAS_GRID:
        cand = bias.clone()
        cand[k] = v
        candidates.append(cand)
    scores = [m["mIoU"] for m in bias_confusions(models[BEST], "val", val_ids, candidates, USE_SMOOTHING)]
    j = int(np.argmax(scores))
    if scores[j] > best_val + BIAS_MIN_GAIN:
        bias, best_val = candidates[j], scores[j]
    print(f"{name}: offset {bias[k].item():+.2f} (val mIoU {best_val:.4f})")
CLASS_BIAS = bias.cpu().numpy().round(3).tolist()

before, after = bias_confusions(models[BEST], "test", RURAL_IDS, [torch.zeros_like(bias), bias], USE_SMOOTHING)
bias_table = pd.DataFrame({"val mIoU": [start_val, best_val], "rural test mIoU": [before["mIoU"], after["mIoU"]],
                           **{f"{c} IoU": [before["IoU"][i], after["IoU"][i]] for i, c in enumerate(CLASS_NAMES)},
                           "Water precision": [before["precision"][5], after["precision"][5]],
                           "Water recall": [before["recall"][5], after["recall"][5]]},
                          index=["no bias", "with bias"]).round(4)
print("class_logit_bias:", dict(zip(CLASS_NAMES, CLASS_BIAS)))
display(bias_table)
bias_table.to_csv(OUT / "class_bias.csv")


## 10. Export for the prototype

`final_model.pt` is the one model to use. The model card holds everything the prototype needs to feed the model exactly as in training (the input format of the 2nd iteration, so `"iteration": 2`), plus:
- `class_logit_bias` (section 9.6), applied by the prototype before the class is chosen;
- `training_years` and `training_new_sites`: the data this model was trained on.

To use it in the prototype: copy `Weights/Final_Iteration/` (or the timestamped folder) next to the other runs and choose it under *Advanced → Other run folder*, or set it as the default (`FINAL_RUN` in `prototype/app.py`) once it has passed the prototype checks (Madagascar highlands, a water-rich site, Thane).


In [ ]:
model_card = {
    "iteration": 2,
    "variant": BEST,
    "n_months": VARIANTS[BEST]["n_months"],
    "in_channels": input_channels(VARIANTS[BEST]),
    "channels_per_month": ["VV_dB", "VH_dB", "VV_minus_VH_dB"],
    "month_window": "target month and the previous n_months-1 months, oldest first (wrapping around the year)",
    "static_channels": STAT_NAMES,
    "channel_order": "monthly channels (oldest month first), then static_channels",
    "s1_preprocessing": S1_PREPROCESSING,
    "stats_definition": "over the 12 months in dB: VV mean, VH mean, (VV-VH) mean, VV 2nd lowest, VV 2nd highest, "
                        "VH 2nd lowest, VH 2nd highest, VH std, share of valid months with VV < "
                        f"{WATER_DB} dB (missing months filled with the pixel mean before sorting); "
                        "standardised with stats_mean/std",
    "stats_mean": STAT_MEAN.tolist(),
    "stats_std": STAT_STD.tolist(),
    "channel_mean": CH_MEAN.tolist(),
    "channel_std": CH_STD.tolist(),
    "temporal_smoothing": ("mean softmax of months t-1, t, t+1 (wrapping) before argmax" if USE_SMOOTHING else "none"),
    "label": "ESA WorldCover 2021 on all pixels; Dynamic World agreement pixels weight 1, others "
             f"{DISAGREE_WEIGHT}; scores on agreement pixels",
    "ema_decay": EMA_DECAY,
    "nan_fill_after_standardisation": 0.0,
    "db_decoding": {"db_min": DB_MIN, "db_step": DB_STEP, "nodata_code": 0},
    "class_names": CLASS_NAMES,
    "class_colors": CLASS_COLORS,
    "patch_size": PATCH,
    "architecture": f"UNetResNet34 ({NOTEBOOK}, same network as Iter2_Test_1_Full)",
    "weights": "final_model.pt",
    "test_rural_mIoU": results_rural[BEST]["raw"]["mIoU"],
    "test_rural_mIoU_smoothed": results_rural[BEST]["smooth"]["mIoU"],
    "test_rural_flicker": results_rural[BEST]["flicker"],
    "test_mIoU": results[BEST]["raw"]["mIoU"],
    "training_years": list(TRAIN_YEARS),
    "training_new_sites": USE_NEW_SITES,
    "class_logit_bias": CLASS_BIAS,
    "class_logit_bias_note": "added to the log of the (smoothed) class probabilities before argmax; tuned on val "
                             f"for {BIAS_CLASSES} ({NOTEBOOK}, section 9.6)",
}
shutil.copy(VARIANT_DIR / f"{BEST}.pt", OUT / "final_model.pt")
json.dump(model_card, open(OUT / "model_card.json", "w"), indent=2)
print({k: v for k, v in model_card.items() if k not in ("stats_mean", "stats_std")})

## 11. Save the run

Everything above is already in `Weights/<RUN_NAME>/`. This cell saves the variant weights and histories once more and zips the folder for one download:

```
Weights/<RUN_NAME>/
  final_model.pt      <- the model to use
  model_card.json     <- how to feed it
  variants/           <- every trained variant + history (report material)
  *.png, *.json       <- curves, confusion, test metrics
```

In [ ]:
for name in models:
    torch.save(models[name].state_dict(), VARIANT_DIR / f"{name}.pt")
    histories[name].to_csv(VARIANT_DIR / f"{name}_history.csv", index=False)
archive = shutil.make_archive(str(OUT), "zip", OUT)
print("saved:", sorted(p.name for p in OUT.iterdir()))
print("archive:", archive)